In [1]:
#判別システム全体
#前処理フェーズ，判別フェーズ

#前処理フェーズ：YOLO
#入力：画像フォルダ
#出力：前処理後の画像

#判別フェーズ
#入力：画像フォルダ
#出力：判別結果

In [2]:
import os 
import cv2
import glob
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
import scipy.stats as stats
import seaborn as sns
import pandas as pd
from sklearn.metrics import roc_curve
import re
import math
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report,precision_score, recall_score, f1_score
import joblib
import time
from ultralytics import YOLO

FlashAttention is not available on this device. Using scaled_dot_product_attention instead.


/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
start = time.time()

In [4]:
data = "maesyori_img"

In [5]:
#複数しいたけ画像の場合
# データフォルダの設定
data = "maesyori_img"

# 入力画像
input_file = "/home/data/maesyori_img/collage_1.jpg"

# 出力フォルダの作成
mask_output_folder = f"/home/data/{data}/mask/"
crop_output_folder = f"/home/data/{data}/crop/"
combined_output_folder = f"/home/data/{data}/combined/"  # 合成画像の出力フォルダ
os.makedirs(mask_output_folder, exist_ok=True)
os.makedirs(crop_output_folder, exist_ok=True)
os.makedirs(combined_output_folder, exist_ok=True)  # 合成フォルダを作成

# モデルの読み込み（物体検出用）
detection_model = YOLO('/home/YOLO/hukusuu_train/datasets/train7/weights/best.pt')
# モデルの読み込み（セグメンテーション用）
segmentation_model = YOLO("/home/YOLO/-327_seg/datasets/train2/weights/best.pt")

# 画像を処理
def process_image(image_path):
    results = detection_model.predict(image_path)  # 物体検出

    # 画像の読み込み
    orig_img = results[0].orig_img
    img_h, img_w, _ = orig_img.shape

    # 4×6のマスに分割
    rows, cols = 4, 6
    cell_h, cell_w = img_h // rows, img_w // cols

    # マス目ごとの bbox を格納するリスト
    cell_bboxes = [[[] for _ in range(cols)] for _ in range(rows)]

    # bbox を対応するマス目に格納
    if results[0].boxes is not None:
        for i, box in enumerate(results[0].boxes.xyxy):
            start_x, start_y, end_x, end_y = map(int, box)
            center_x, center_y = (start_x + end_x) // 2, (start_y + end_y) // 2
            
            row_idx = min(center_y // cell_h, rows - 1)
            col_idx = min(center_x // cell_w, cols - 1)
            
            cell_bboxes[row_idx][col_idx].append((start_x, start_y, end_x, end_y))

    # 出力フォルダを作成
    base_name = os.path.splitext(os.path.basename(image_path))[0]
    image_output_folder = os.path.join(crop_output_folder, base_name)
    mask_output_folder_specific = os.path.join(mask_output_folder, base_name)
    combined_output_folder_specific = os.path.join(combined_output_folder, base_name)
    os.makedirs(image_output_folder, exist_ok=True)
    os.makedirs(mask_output_folder_specific, exist_ok=True)
    os.makedirs(combined_output_folder_specific, exist_ok=True)  # 合成フォルダを作成

    # マス目順に bbox を保存
    for row in range(rows):
        for col in range(cols):
            if not cell_bboxes[row][col]:  # bboxがない場合はスキップ
                continue
            for idx, (sx, sy, ex, ey) in enumerate(cell_bboxes[row][col]):
                clip_img = orig_img[sy:ey, sx:ex]
                clip_filename = os.path.join(image_output_folder, f"clip_{row+1}_{col+1}.jpg")
                cv2.imwrite(clip_filename, clip_img)

                # セグメンテーション用のマスク作成
                mask_results = segmentation_model.predict(clip_img)
                if mask_results and mask_results[0].masks is not None:
                    mask = mask_results[0].masks.data[0].cpu().numpy()  # CUDA -> CPU に転送して NumPy に変換
                    mask = (mask * 255).astype(np.uint8)  # 0-1 を 0-255 に変換

                    # マスク画像を切り抜き画像のサイズにリサイズ
                    mask_resized = cv2.resize(mask, (clip_img.shape[1], clip_img.shape[0]))

                    # マスク画像を保存
                    mask_filename = os.path.join(mask_output_folder_specific, f"mask_{row+1}_{col+1}.jpg")
                    cv2.imwrite(mask_filename, mask_resized)

                    # マスクと切り抜き画像を掛け合わせる（積を取る）
                    # clip_img_rgb = cv2.cvtColor(clip_img, cv2.COLOR_BGR2RGB)  # RGB形式に変換
                    mask_rgb = cv2.cvtColor(mask_resized, cv2.COLOR_GRAY2BGR)  # グレースケールのマスクをRGBに変換

                    # 切り抜き画像とリサイズしたマスク画像の積
                    combined_img = cv2.bitwise_and(clip_img, mask_rgb)  # 切り抜き画像とマスクの積

                    # 合成画像を保存
                    combined_filename = os.path.join(combined_output_folder_specific, f"combined_{row+1}_{col+1}.jpg")
                    cv2.imwrite(combined_filename, combined_img)

# 画像1枚のみ処理
process_image(input_file)



image 1/1 /home/data/maesyori_img/collage_1.jpg: 320x640 24 shiitake_bboxs, 23.1ms
Speed: 1.8ms preprocess, 23.1ms inference, 0.4ms postprocess per image at shape (1, 3, 320, 640)

0: 640x640 1 shiitake, 35.9ms
Speed: 0.9ms preprocess, 35.9ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 608x640 1 shiitake, 28.5ms
Speed: 1.0ms preprocess, 28.5ms inference, 0.8ms postprocess per image at shape (1, 3, 608, 640)

0: 640x576 1 shiitake, 28.4ms
Speed: 0.9ms preprocess, 28.4ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 576)

0: 640x640 1 shiitake, 23.4ms
Speed: 1.3ms preprocess, 23.4ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 shiitake, 23.0ms
Speed: 1.3ms preprocess, 23.0ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x608 1 shiitake, 31.7ms
Speed: 1.0ms preprocess, 31.7ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 608)

0: 640x608 1 shiitake, 20.4ms
Speed: 0.9ms preproc

In [ ]:
import hida
import keijo
import size_module
#判別フェーズ
#特徴量抽出
hida_tapple = hida.Hida_folder(base_dir="/home/data/maesyori_img")
result_hida = hida_tapple.run_all()
size_tapple = size_module.Size_folder(base_dir="/home/data/maesyori_img")
result_size = size_tapple.count_white_pixels()
keijo_tapple = keijo.Keijo_folder(base_dir="/home/data/maesyori_img")
result_keijo = keijo_tapple.run()

探索対象フォルダ: /home/data/maesyori_img/mask/collage_1
📂 フォルダ: /home/data/maesyori_img/mask/collage_1 に画像 24 枚
[('mask_1_1.jpg', 0.5183052241875771), ('mask_1_2.jpg', 0.6670925732081193), ('mask_1_3.jpg', 0.5700113038590996), ('mask_1_4.jpg', 0.7341753452987665), ('mask_1_5.jpg', 0.5472011766516734), ('mask_1_6.jpg', 0.6490946210864547), ('mask_2_1.jpg', 0.513392761709314), ('mask_2_2.jpg', 0.5856268856386403), ('mask_2_3.jpg', 0.5224796376180154), ('mask_2_4.jpg', 0.513080528832104), ('mask_2_5.jpg', 0.5106721746505075), ('mask_2_6.jpg', 0.5111390645171968), ('mask_3_1.jpg', 0.5345710538389882), ('mask_3_2.jpg', 0.5079078819331819), ('mask_3_3.jpg', 0.8192692059393157), ('mask_3_4.jpg', 0.6943605425977682), ('mask_3_5.jpg', 0.5020211221497908), ('mask_3_6.jpg', 0.7001554649302661), ('mask_4_1.jpg', 0.5185988545769356), ('mask_4_2.jpg', 0.5274708244299904), ('mask_4_3.jpg', 0.5121353307008197), ('mask_4_4.jpg', 0.5836307714001181), ('mask_4_5.jpg', 0.5105739519186518), ('mask_4_6.jpg', 0.707

In [7]:
import pandas as pd
dict_hida =dict(result_hida)
dict_size = dict(result_size)
dict_keijo = dict(result_keijo)
# 結果を結合
merged = []
for filename in dict_hida.keys() & dict_size.keys() & dict_keijo.keys():
    merged.append({
        'filename': filename,
        'MSE': dict_keijo[filename],
        'size_count': dict_size[filename],
        'R': dict_hida[filename]
    })

In [9]:
df = pd.DataFrame(merged)

In [11]:
# === モデルとスケーラーの読み込み ===
model_path = "svm_model.pkl"
scaler_path = "scaler.pkl"

svm_model = joblib.load(model_path)
scaler = joblib.load(scaler_path)

# === 新しいデータの読み込み ===
df = pd.DataFrame(merged)

# === 特徴量の抽出と標準化 ===
X_new = df[["MSE", "size_count", "R"]]  # 学習時と同じ特徴量を使用
X_new = scaler.transform(X_new)  # 標準化

# === 予測 ===
y_pred_new = svm_model.predict(X_new)

# 結果をDataFrameに追加
df["Predicted_Label"] = y_pred_new

# 予測結果の確認
print([["MSE", "size_count", "R", "Predicted_Label"]])

# CSVとして保存（オプション）
df.to_csv(f"/home/data/{data}/predicted_results.csv", index=False)


[['MSE', 'size_count', 'R', 'Predicted_Label']]


/usr/local/lib/python3.11/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.11/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [12]:
end = time.time()
print(f"Total time: {end - start:.2f} sec")

Total time: 15.36 sec


In [ ]:
# import hida
# a = hida.Hida_file(img_path = "/home/data/maesyori_img/combined/collage_1/combined_1_1.jpg", mask_path = "/home/data/maesyori_img/mask/collage_1/mask_1_1.jpg")
# result = a.calculate_R()
# print(result)

0.5183052241875771


In [ ]:
# import hida
# a = hida.Hida_folder(base_dir="/home/data/maesyori_img")
# result = a.run_all()
# print(result)

[('mask_1_1.jpg', 0.5183052241875771), ('mask_1_2.jpg', 0.6670925732081193), ('mask_1_3.jpg', 0.5700113038590996), ('mask_1_4.jpg', 0.7341753452987665), ('mask_1_5.jpg', 0.5472011766516734), ('mask_1_6.jpg', 0.6490946210864547), ('mask_2_1.jpg', 0.513392761709314), ('mask_2_2.jpg', 0.5856268856386403), ('mask_2_3.jpg', 0.5224796376180154), ('mask_2_4.jpg', 0.513080528832104), ('mask_2_5.jpg', 0.5106721746505075), ('mask_2_6.jpg', 0.5111390645171968), ('mask_3_1.jpg', 0.5345710538389882), ('mask_3_2.jpg', 0.5079078819331819), ('mask_3_3.jpg', 0.8192692059393157), ('mask_3_4.jpg', 0.6943605425977682), ('mask_3_5.jpg', 0.5020211221497908), ('mask_3_6.jpg', 0.7001554649302661), ('mask_4_1.jpg', 0.5185988545769356), ('mask_4_2.jpg', 0.5274708244299904), ('mask_4_3.jpg', 0.5121353307008197), ('mask_4_4.jpg', 0.5836307714001181), ('mask_4_5.jpg', 0.5105739519186518), ('mask_4_6.jpg', 0.7076615835037954)]


In [ ]:
# import size

# a = size.Size_file("/home/data/maesyori_img/mask/collage_1/mask_1_1.jpg")
# result = a.count_white_pixels()
# print(result)

121512


In [ ]:
# import size_module
# import os
# import cv2
# import numpy as np
# b = size_module.Size_folder(base_dir="/home/data/maesyori_img")
# result = b.count_white_pixels()
# print(result)

探索対象フォルダ: /home/data/maesyori_img/mask/collage_1
[('mask_2_5.jpg', 185937), ('mask_3_6.jpg', 225896), ('mask_1_1.jpg', 121512), ('mask_3_1.jpg', 218938), ('mask_1_2.jpg', 272363), ('mask_2_1.jpg', 143241), ('mask_4_5.jpg', 139556), ('mask_4_4.jpg', 230675), ('mask_4_6.jpg', 209825), ('mask_4_3.jpg', 146043), ('mask_1_3.jpg', 161240), ('mask_3_3.jpg', 196777), ('mask_2_3.jpg', 141655), ('mask_2_6.jpg', 153869), ('mask_2_4.jpg', 174068), ('mask_3_5.jpg', 143178), ('mask_1_5.jpg', 296909), ('mask_4_1.jpg', 160475), ('mask_1_4.jpg', 261250), ('mask_2_2.jpg', 181757), ('mask_4_2.jpg', 216597), ('mask_3_4.jpg', 243589), ('mask_1_6.jpg', 175273), ('mask_3_2.jpg', 156780)]


In [ ]:
# import keijo
# analyzer = keijo.Keijo_file("/home/data/maesyori_img/mask/collage_1/mask_1_1.jpg")
# result = analyzer.run()
# print(result)

2773.227906976744


In [ ]:
# import keijo
# analyzer = keijo.Keijo_folder(base_dir="/home/data/maesyori_img")
# result = analyzer.run()
# print(result)

📂 フォルダ: /home/data/maesyori_img/mask/collage_1 に画像 24 枚
[('mask_2_5.jpg', 3171.5036363636364), ('mask_3_6.jpg', 8475.136294027565), ('mask_1_1.jpg', 2773.227906976744), ('mask_3_1.jpg', 2278.6748681898066), ('mask_1_2.jpg', 5995.815831987076), ('mask_2_1.jpg', 1223.6505263157894), ('mask_4_5.jpg', 2636.8630136986303), ('mask_4_4.jpg', 927.3727121464226), ('mask_4_6.jpg', 8314.751893939394), ('mask_4_3.jpg', 2531.116331096197), ('mask_1_3.jpg', 984.375486381323), ('mask_3_3.jpg', 1793.2692307692307), ('mask_2_3.jpg', 782.7086092715232), ('mask_2_6.jpg', 806.353305785124), ('mask_2_4.jpg', 1019.6167315175097), ('mask_3_5.jpg', 1117.9674620390456), ('mask_1_5.jpg', 412.14153846153846), ('mask_4_1.jpg', 199.8483606557377), ('mask_1_4.jpg', 2073.7353846153846), ('mask_2_2.jpg', 1273.4429133858268), ('mask_4_2.jpg', 843.8719298245614), ('mask_3_4.jpg', 3510.9641185647424), ('mask_1_6.jpg', 2876.113345521024), ('mask_3_2.jpg', 1752.6523605150214)]
